In [4]:
# ============================================================
# 0. 라이브러리 & 경로 설정
# ============================================================
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

TARGET_COLS = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]
CAPACITY_KWH = {"kpx_group_1": 21600, "kpx_group_2": 21600, "kpx_group_3": 21000}
ID_COLS = ["forecast_kst_dtm", "data_available_kst_dtm", "grid_id", "latitude", "longitude"]
CORR_VAL_START = pd.Timestamp("2024-01-01")


# ============================================================
# 1. 데이터 로더 (가공 없음 — 원본 컬럼 전부 사용, 격자는 단순 전체평균만)
# ============================================================
def load_csv(path):
    df = pd.read_csv(path, encoding="utf-8-sig")
    df["forecast_kst_dtm"] = pd.to_datetime(df["forecast_kst_dtm"])
    return df


def simple_mean_aggregate(df, prefix):
    """가공 없이 모든 원본 컬럼을 그대로, 격자만 단순평균."""
    value_cols = [c for c in df.columns if c not in ID_COLS]
    agg = df.groupby("forecast_kst_dtm")[value_cols].mean()
    agg.columns = [f"{prefix}_{c}" for c in agg.columns]
    return agg.reset_index()


def calendar_features(dt_series):
    dt = pd.to_datetime(dt_series)
    out = pd.DataFrame(index=dt.index)
    out["hour_sin"] = np.sin(2 * np.pi * dt.dt.hour / 24)
    out["hour_cos"] = np.cos(2 * np.pi * dt.dt.hour / 24)
    out["month_sin"] = np.sin(2 * np.pi * dt.dt.month / 12)
    out["month_cos"] = np.cos(2 * np.pi * dt.dt.month / 12)
    return out


def build_raw_dataset_for_group(split, group, labels=None):
    ldaps = load_csv((TRAIN_DIR if split == "train" else TEST_DIR) / f"ldaps_{split}.csv")
    gfs = load_csv((TRAIN_DIR if split == "train" else TEST_DIR) / f"gfs_{split}.csv")

    # 컬럼 제거 없음 - 원본 전부 사용 (50m max/min 포함)
    ldaps_agg = simple_mean_aggregate(ldaps, "ldaps")
    gfs_agg = simple_mean_aggregate(gfs, "gfs")

    df = ldaps_agg.merge(gfs_agg, on="forecast_kst_dtm", how="inner")
    df = pd.concat([df, calendar_features(df["forecast_kst_dtm"])], axis=1)

    if labels is not None:
        y = labels[["kst_dtm", group]].rename(columns={"kst_dtm": "forecast_kst_dtm", group: "target"})
        df = df.merge(y, on="forecast_kst_dtm", how="left").dropna(subset=["target"])

    df = df.sort_values("forecast_kst_dtm").reset_index(drop=True)
    feat_cols = [c for c in df.columns if c not in ("forecast_kst_dtm", "target")]
    df[feat_cols] = df[feat_cols].interpolate(method="linear", limit_direction="both")
    return df


# ============================================================
# 2. 데이터 로드 실행
# ============================================================
train_labels = pd.read_csv(TRAIN_DIR / "train_labels.csv", encoding="utf-8-sig")
train_labels["kst_dtm"] = pd.to_datetime(train_labels["kst_dtm"])

train_datasets = {}
test_datasets = {}
for g in TARGET_COLS:
    train_datasets[g] = build_raw_dataset_for_group("train", g, train_labels)
    test_datasets[g] = build_raw_dataset_for_group("test", g, None)

RAW_FEATS = [c for c in train_datasets["kpx_group_1"].columns if c not in ("forecast_kst_dtm", "target")]

for g in TARGET_COLS:
    df = train_datasets[g]
    tr_check = df[df["forecast_kst_dtm"] < CORR_VAL_START]
    va_check = df[df["forecast_kst_dtm"] >= CORR_VAL_START]
    print(f"[{g}] train shape={train_datasets[g].shape}, test shape={test_datasets[g].shape}")
    print(f"       train range: {tr_check['forecast_kst_dtm'].min()} ~ {tr_check['forecast_kst_dtm'].max()} (n={len(tr_check)})")
    print(f"       val range:   {va_check['forecast_kst_dtm'].min()} ~ {va_check['forecast_kst_dtm'].max()} (n={len(va_check)})")
print(f"\n전체 피처 수: {len(RAW_FEATS)}")


# ============================================================
# 3. 평가지표 함수 (대회 공식 metric)
# ============================================================
def group_score(actual, forecast, capacity):
    valid = actual >= capacity * 0.10
    actual_v, forecast_v = actual[valid], forecast[valid]
    error_rate = np.abs(forecast_v - actual_v) / capacity
    nmae = np.mean(error_rate)
    unit_price = np.select([error_rate <= 0.06, error_rate <= 0.08], [4.0, 3.0], default=0.0)
    earned = np.sum(actual_v * unit_price)
    maxset = np.sum(actual_v * 4.0)
    ficr = earned / maxset
    return 0.5 * (1 - nmae) + 0.5 * ficr, nmae, ficr


# ============================================================
# 4. 6개 모델 비교 + top2/top3 앙상블
# ============================================================
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor


def build_base_models():
    return {
        "RandomForest": RandomForestRegressor(n_estimators=300, max_depth=16, random_state=42, n_jobs=-1),
        "HistGB": HistGradientBoostingRegressor(max_iter=300, max_depth=None, random_state=42),  # HistGB는 None이면 자동으로 깊이 제한 없이 leaf 수로 제어
        "LightGBM": LGBMRegressor(n_estimators=300, max_depth=-1, num_leaves=63, learning_rate=0.05, random_state=42, verbose=-1),
        "XGBoost": XGBRegressor(n_estimators=300, max_depth=10, learning_rate=0.05, random_state=42, verbosity=0),
        "CatBoost": CatBoostRegressor(iterations=300, depth=10, verbose=False, random_state=42, allow_writing_files=False),
        "MLP": make_pipeline(
            StandardScaler(),
            MLPRegressor(hidden_layer_sizes=(64, 32), early_stopping=True, random_state=42, max_iter=2000),
        ),
    }


all_results = []
for g in TARGET_COLS:
    df = train_datasets[g]
    tr = df[df["forecast_kst_dtm"] < CORR_VAL_START]
    va = df[df["forecast_kst_dtm"] >= CORR_VAL_START].copy()
    capacity = CAPACITY_KWH[g]
    y_va = va["target"].to_numpy(dtype=float)

    print(f"===== {g} ===== train={len(tr)} val={len(va)}")
    preds = {}
    for name, model in build_base_models().items():
        model.fit(tr[RAW_FEATS], tr["target"])
        pred = np.clip(model.predict(va[RAW_FEATS]), 0, capacity)
        preds[name] = pred
        score, nmae, ficr = group_score(y_va, pred, capacity)
        print(f"  {name:15s} score={score:.4f} nmae={nmae:.4f} ficr={ficr:.4f}")
        all_results.append({"group": g, "model": name, "score": score, "nmae": nmae, "ficr": ficr})

    ranked = sorted(
        preds.keys(),
        key=lambda n: [r["score"] for r in all_results if r["group"] == g and r["model"] == n][0],
        reverse=True,
    )
    for k, label in [(2, "Ensemble_top2"), (3, "Ensemble_top3")]:
        ens = np.clip(np.mean([preds[n] for n in ranked[:k]], axis=0), 0, capacity)
        score, nmae, ficr = group_score(y_va, ens, capacity)
        print(f"  {label:15s} ({'+'.join(ranked[:k])}) score={score:.4f} nmae={nmae:.4f} ficr={ficr:.4f}")
        all_results.append({"group": g, "model": label, "score": score, "nmae": nmae, "ficr": ficr})
    print()

results_df = pd.DataFrame(all_results)
summary = results_df.groupby("model")[["score", "nmae", "ficr"]].mean().round(4).sort_values("score", ascending=False)
print("=" * 60)
print("전체 요약 — 원본 변수 전부 + 캘린더 (가공 없음)")
print("=" * 60)
print(summary)

[kpx_group_1] train shape=(26200, 71), test shape=(8760, 70)
       train range: 2022-01-01 01:00:00 ~ 2023-12-31 23:00:00 (n=17421)
       val range:   2024-01-01 00:00:00 ~ 2025-01-01 00:00:00 (n=8779)
[kpx_group_2] train shape=(26201, 71), test shape=(8760, 70)
       train range: 2022-01-01 01:00:00 ~ 2023-12-31 23:00:00 (n=17422)
       val range:   2024-01-01 00:00:00 ~ 2025-01-01 00:00:00 (n=8779)
[kpx_group_3] train shape=(17538, 71), test shape=(8760, 70)
       train range: 2023-01-01 01:00:00 ~ 2023-12-31 23:00:00 (n=8759)
       val range:   2024-01-01 00:00:00 ~ 2025-01-01 00:00:00 (n=8779)

전체 피처 수: 69
===== kpx_group_1 ===== train=17421 val=8779
  RandomForest    score=0.5807 nmae=0.1291 ficr=0.2904
  HistGB          score=0.6033 nmae=0.1267 ficr=0.3334
  LightGBM        score=0.6042 nmae=0.1264 ficr=0.3347
  XGBoost         score=0.5902 nmae=0.1295 ficr=0.3099
  CatBoost        score=0.5920 nmae=0.1295 ficr=0.3134
  MLP             score=0.5934 nmae=0.1321 ficr=0.3190
 

In [11]:
# ============================================================
# 0. 그룹별 격자 설정 확정
#    그룹1: 인접4개[5,6,10,11] - 실험 결과 확실한 개선(+0.006) 확인됨
#    그룹2, 3: 16개 단순평균 - 격자방식 무관/기존이 최선이었음
# ============================================================
GROUP_ADJACENT4_GRIDS = {
    "kpx_group_1": [5, 6, 10, 11],
    "kpx_group_2": [6, 7, 11, 12],
    "kpx_group_3": [6, 7, 11, 12],
}
GFS_GRID = 5

GRID_CONFIG = {
    "kpx_group_1": "adjacent4",
    "kpx_group_2": "all16",
    "kpx_group_3": "all16",
}


def load_csv(path):
    df = pd.read_csv(path, encoding="utf-8-sig")
    df["forecast_kst_dtm"] = pd.to_datetime(df["forecast_kst_dtm"])
    return df


def aggregate_grids(df, grids, prefix):
    sub = df[df["grid_id"].isin(grids)]
    value_cols = [c for c in sub.columns if c not in ID_COLS]
    agg = sub.groupby("forecast_kst_dtm")[value_cols].mean()
    agg.columns = [f"{prefix}_{c}" for c in agg.columns]
    return agg.reset_index()


def build_dataset_with_grid(split, group, labels=None):
    ldaps = load_csv((TRAIN_DIR if split == "train" else TEST_DIR) / f"ldaps_{split}.csv")
    gfs = load_csv((TRAIN_DIR if split == "train" else TEST_DIR) / f"gfs_{split}.csv")

    grid_mode = GRID_CONFIG[group]
    ldaps_grids = GROUP_ADJACENT4_GRIDS[group] if grid_mode == "adjacent4" else list(range(1, 17))

    ldaps_agg = aggregate_grids(ldaps, ldaps_grids, "ldaps")
    gfs_agg = aggregate_grids(gfs, [GFS_GRID], "gfs")

    df = ldaps_agg.merge(gfs_agg, on="forecast_kst_dtm", how="inner")
    df = pd.concat([df, calendar_features(df["forecast_kst_dtm"])], axis=1)

    if labels is not None:
        y = labels[["kst_dtm", group]].rename(columns={"kst_dtm": "forecast_kst_dtm", group: "target"})
        df = df.merge(y, on="forecast_kst_dtm", how="left").dropna(subset=["target"])

    df = df.sort_values("forecast_kst_dtm").reset_index(drop=True)
    feat_cols = [c for c in df.columns if c not in ("forecast_kst_dtm", "target")]
    df[feat_cols] = df[feat_cols].interpolate(method="linear", limit_direction="both")
    return df


# 그룹별 격자설정 반영해서 train_datasets, test_datasets 재생성
train_datasets = {}
test_datasets = {}
for g in TARGET_COLS:
    train_datasets[g] = build_dataset_with_grid("train", g, train_labels)
    test_datasets[g] = build_dataset_with_grid("test", g, None)

RAW_FEATS = [c for c in train_datasets["kpx_group_1"].columns if c not in ("forecast_kst_dtm", "target")]

for g in TARGET_COLS:
    print(f"[{g}] grid={GRID_CONFIG[g]:10s} train shape={train_datasets[g].shape}, test shape={test_datasets[g].shape}")


# ============================================================
# Baseline 확정: 전체피처(RAW_FEATS) + 그룹별 최적 학습기간 + 그룹별 top2 모델 (개별로 각각 평가)
# ============================================================

TRAIN_PERIOD_START = {
    "kpx_group_1": "2023-01-01",  # 2023만: score 0.6091 > 2022+2023: 0.6042
    "kpx_group_2": "2022-01-01",  # 2022+2023: score 0.6428 > 2023만: 0.6241
    "kpx_group_3": "2023-01-01",  # 원래부터 2023년만 존재
}

TOP2_MODELS = {
    "kpx_group_1": ["LightGBM", "HistGB"],
    "kpx_group_2": ["HistGB", "LightGBM"],
    "kpx_group_3": ["MLP", "RandomForest"],
}

def make_model(name):
    if name == "RandomForest":
        return RandomForestRegressor(n_estimators=300, max_depth=16, random_state=42, n_jobs=-1)
    if name == "HistGB":
        return HistGradientBoostingRegressor(max_iter=300, max_depth=None, random_state=42)
    if name == "LightGBM":
        return LGBMRegressor(n_estimators=300, max_depth=-1, num_leaves=63, learning_rate=0.05, random_state=42, verbose=-1)
    if name == "XGBoost":
        return XGBRegressor(n_estimators=300, max_depth=10, learning_rate=0.05, random_state=42, verbosity=0)
    if name == "CatBoost":
        return CatBoostRegressor(iterations=300, depth=10, verbose=False, random_state=42, allow_writing_files=False)
    if name == "MLP":
        return make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(64, 32), early_stopping=True, random_state=42, max_iter=2000))
    raise ValueError(name)


# ============================================================
# Baseline 학습 + val(2024) 평가 — 그룹별 top2 모델을 각각 독립적으로 평가
# ============================================================
baseline_models = {}       # (group, model_name) -> fitted_model
baseline_val_results = {}  # (group, model_name) -> val df (pred 포함)
baseline_summary = []

for g in TARGET_COLS:
    df = train_datasets[g]
    tr = df[(df["forecast_kst_dtm"] >= TRAIN_PERIOD_START[g]) & (df["forecast_kst_dtm"] < CORR_VAL_START)]
    va_base = df[df["forecast_kst_dtm"] >= CORR_VAL_START]
    capacity = CAPACITY_KWH[g]
    y_va = va_base["target"].to_numpy(dtype=float)

    for name in TOP2_MODELS[g]:
        model = make_model(name)
        model.fit(tr[RAW_FEATS], tr["target"])

        va = va_base.copy()
        va["pred"] = np.clip(model.predict(va[RAW_FEATS]), 0, capacity)
        pred_va = va["pred"].to_numpy(dtype=float)
        score, nmae, ficr = group_score(y_va, pred_va, capacity)

        baseline_models[(g, name)] = model
        baseline_val_results[(g, name)] = va
        baseline_summary.append({
            "group": g, "model": name, "grid": GRID_CONFIG[g], "train_start": TRAIN_PERIOD_START[g],
            "n_train": len(tr), "score": score, "nmae": nmae, "ficr": ficr,
        })

        print(f"[{g}] model={name:15s} grid={GRID_CONFIG[g]:10s} train_start={TRAIN_PERIOD_START[g]} n_train={len(tr):6d} "
              f"-> score={score:.4f} nmae={nmae:.4f} ficr={ficr:.4f}")
    print()

baseline_summary_df = pd.DataFrame(baseline_summary)
print("=" * 70)
print(baseline_summary_df)
print()
print("그룹별 최고 score 기준 평균:", baseline_summary_df.groupby("group")["score"].max().mean().round(4))

[kpx_group_1] grid=adjacent4  train shape=(26200, 71), test shape=(8760, 70)
[kpx_group_2] grid=all16      train shape=(26201, 71), test shape=(8760, 70)
[kpx_group_3] grid=all16      train shape=(17538, 71), test shape=(8760, 70)
[kpx_group_1] model=LightGBM        grid=adjacent4  train_start=2023-01-01 n_train=  8757 -> score=0.6113 nmae=0.1262 ficr=0.3488
[kpx_group_1] model=HistGB          grid=adjacent4  train_start=2023-01-01 n_train=  8757 -> score=0.6195 nmae=0.1268 ficr=0.3657

[kpx_group_2] model=HistGB          grid=all16      train_start=2022-01-01 n_train= 17422 -> score=0.6439 nmae=0.1240 ficr=0.4119
[kpx_group_2] model=LightGBM        grid=all16      train_start=2022-01-01 n_train= 17422 -> score=0.6492 nmae=0.1244 ficr=0.4227

[kpx_group_3] model=MLP             grid=all16      train_start=2023-01-01 n_train=  8759 -> score=0.5534 nmae=0.1587 ficr=0.2655
[kpx_group_3] model=RandomForest    grid=all16      train_start=2023-01-01 n_train=  8759 -> score=0.5503 nmae=0.1512

In [13]:
# ============================================================
# 변수명 한글 매핑
# ============================================================
VAR_KOREAN_NAME = {
    "ldaps_heightAboveGround_10_10u": "LDAPS 10m U바람", "ldaps_heightAboveGround_10_10v": "LDAPS 10m V바람",
    "ldaps_heightAboveGround_50_50MUmax": "LDAPS 50m U최댓값", "ldaps_heightAboveGround_50_50MUmin": "LDAPS 50m U최솟값",
    "ldaps_heightAboveGround_50_50MVmax": "LDAPS 50m V최댓값", "ldaps_heightAboveGround_50_50MVmin": "LDAPS 50m V최솟값",
    "ldaps_heightAboveGround_5_XBLWS": "LDAPS 5m X경계층바람", "ldaps_heightAboveGround_5_YBLWS": "LDAPS 5m Y경계층바람",
    "ldaps_heightAboveGround_2_t": "LDAPS 2m기온", "ldaps_heightAboveGround_2_dpt": "LDAPS 2m이슬점",
    "ldaps_heightAboveGround_2_r": "LDAPS 2m상대습도", "ldaps_heightAboveGround_2_q": "LDAPS 2m비습",
    "ldaps_surface_0_sp": "LDAPS 지표기압", "ldaps_meanSea_0_prmsl": "LDAPS 해면기압",
    "ldaps_etc_0_blh": "LDAPS 경계층높이", "ldaps_surface_0_NDNSW": "LDAPS 순단파복사",
    "ldaps_surface_0_NDNLW": "LDAPS 순장파복사", "ldaps_heightAboveGround_2_SWDIR": "LDAPS 직달단파",
    "ldaps_heightAboveGround_2_SWDIF": "LDAPS 산란단파", "ldaps_etc_0_hcc": "LDAPS 상층운량",
    "ldaps_etc_0_mcc": "LDAPS 중층운량", "ldaps_etc_0_lcc": "LDAPS 하층운량", "ldaps_etc_0_VLCDC": "LDAPS 매우낮은운량",
    "ldaps_surface_0_avg_lsprate": "LDAPS 평균강수율", "ldaps_surface_0_lssrate": "LDAPS 대규모강설률",
    "ldaps_surface_0_ncpcp": "LDAPS 비대류강수량", "ldaps_surface_0_snol": "LDAPS 적설관련",
    "ldaps_surface_0_SNOM": "LDAPS 융설량", "ldaps_surface_0_lsm": "LDAPS 육지해양마스크", "ldaps_surface_0_h": "LDAPS 지표고도",
    "gfs_heightAboveGround_10_10u": "GFS 10m U바람", "gfs_heightAboveGround_10_10v": "GFS 10m V바람",
    "gfs_heightAboveGround_80_u": "GFS 80m U바람", "gfs_heightAboveGround_80_v": "GFS 80m V바람",
    "gfs_heightAboveGround_100_100u": "GFS 100m U바람(허브)", "gfs_heightAboveGround_100_100v": "GFS 100m V바람(허브)",
    "gfs_heightAboveGround_2_2t": "GFS 2m기온", "gfs_heightAboveGround_2_2d": "GFS 2m이슬점",
    "gfs_heightAboveGround_2_2r": "GFS 2m상대습도", "gfs_heightAboveGround_2_2sh": "GFS 2m비습",
    "gfs_planetaryBoundaryLayer_0_u": "GFS 경계층U바람", "gfs_planetaryBoundaryLayer_0_v": "GFS 경계층V바람",
    "gfs_planetaryBoundaryLayer_0_VRATE": "GFS 경계층수직속도", "gfs_surface_0_dswrf": "GFS 하향단파플럭스",
    "gfs_surface_0_dlwrf": "GFS 하향장파플럭스", "gfs_surface_0_prate": "GFS 강수율", "gfs_surface_0_tp": "GFS 총강수량",
    "gfs_surface_0_sp": "GFS 지표기압", "gfs_meanSea_0_prmsl": "GFS 해면기압", "gfs_surface_0_gust": "GFS 돌풍",
    "gfs_lowCloudLayer_0_lcc": "GFS 하층운량", "gfs_middleCloudLayer_0_mcc": "GFS 중층운량",
    "gfs_highCloudLayer_0_hcc": "GFS 상층운량", "gfs_atmosphere_0_tcc": "GFS 전운량",
    "gfs_isobaricInhPa_850_t": "GFS 850hPa기온", "gfs_isobaricInhPa_850_u": "GFS 850hPa U바람",
    "gfs_isobaricInhPa_850_v": "GFS 850hPa V바람", "gfs_isobaricInhPa_850_r": "GFS 850hPa습도",
    "gfs_isobaricInhPa_700_t": "GFS 700hPa기온", "gfs_isobaricInhPa_700_u": "GFS 700hPa U바람",
    "gfs_isobaricInhPa_700_v": "GFS 700hPa V바람", "gfs_isobaricInhPa_500_gh": "GFS 500hPa지위고도",
    "gfs_isobaricInhPa_500_t": "GFS 500hPa기온", "gfs_isobaricInhPa_500_u": "GFS 500hPa U바람",
    "gfs_isobaricInhPa_500_v": "GFS 500hPa V바람",
    "hour_sin": "시간(sin)", "hour_cos": "시간(cos)", "month_sin": "월(sin)", "month_cos": "월(cos)",
}

# ============================================================
# 1. 오차구간(zone1~4)별 z-score
#    - 각 그룹×모델 블록마다 zone1~4 표본수/평균오차율을 전부 출력
#    - 변수별 표에도 zone1~4 값을 전부 나란히 표시
# ============================================================
zone_screening_results = {}

for g in TARGET_COLS:
    for model_name in TOP2_MODELS[g]:
        va = baseline_val_results[(g, model_name)].copy()
        va["error_rate"] = np.abs(va["pred"] - va["target"]) / CAPACITY_KWH[g]

        zone1 = va[va["error_rate"] <= 0.06]
        zone2 = va[(va["error_rate"] > 0.06) & (va["error_rate"] <= 0.08)]
        zone3 = va[(va["error_rate"] > 0.08) & (va["error_rate"] <= 0.10)]
        zone4 = va[va["error_rate"] > 0.10]

        print(f"===== {g} / {model_name} =====")
        print(f"  zone1(<=6%):       n={len(zone1):5d}  평균오차율={zone1['error_rate'].mean()*100:5.2f}%")
        print(f"  zone2(6~8%):       n={len(zone2):5d}  평균오차율={zone2['error_rate'].mean()*100:5.2f}%")
        print(f"  zone3(8~10%,타겟): n={len(zone3):5d}  평균오차율={zone3['error_rate'].mean()*100:5.2f}%")
        print(f"  zone4(>10%):       n={len(zone4):5d}  평균오차율={zone4['error_rate'].mean()*100:5.2f}%")

        rows = []
        for c in RAW_FEATS:
            z1, z2, z3, z4 = zone1[c].mean(), zone2[c].mean(), zone3[c].mean(), zone4[c].mean()
            std = va[c].std()
            z_diff_13 = (z3 - z1) / std if std > 0 else 0
            rows.append({
                "변수": VAR_KOREAN_NAME.get(c, c),
                "zone1(≤6%)": round(z1, 3),
                "zone2(6~8%)": round(z2, 3),
                "zone3(8~10%,타겟)": round(z3, 3),
                "zone4(>10%)": round(z4, 3),
                "z_score(1vs3)": round(z_diff_13, 3),
            })

        result = pd.DataFrame(rows).sort_values("z_score(1vs3)", key=abs, ascending=False)
        zone_screening_results[(g, model_name)] = result
        print(result.head(15).to_string(index=False))
        print()


# ============================================================
# 2. 변수 간 상관관계 분석 — 그룹별로 (격자방식이 그룹1만 다르므로 그룹별 실행)
# ============================================================
for g in TARGET_COLS:
    df = train_datasets[g]
    corr = df[RAW_FEATS].corr()

    corr_pairs = []
    for i in range(len(RAW_FEATS)):
        for j in range(i + 1, len(RAW_FEATS)):
            c_val = corr.iloc[i, j]
            if abs(c_val) >= 0.9:
                corr_pairs.append({
                    "변수1": VAR_KOREAN_NAME.get(RAW_FEATS[i], RAW_FEATS[i]),
                    "변수2": VAR_KOREAN_NAME.get(RAW_FEATS[j], RAW_FEATS[j]),
                    "상관계수": round(c_val, 3),
                })

    corr_df = pd.DataFrame(corr_pairs).sort_values("상관계수", key=abs, ascending=False)
    print(f"===== {g}: |상관계수| >= 0.9 인 변수쌍 (총 {len(corr_df)}쌍) =====")
    print(corr_df.to_string(index=False))
    print()

===== kpx_group_1 / LightGBM =====
  zone1(<=6%):       n= 4199  평균오차율= 2.48%
  zone2(6~8%):       n=  723  평균오차율= 6.96%
  zone3(8~10%,타겟): n=  639  평균오차율= 8.96%
  zone4(>10%):       n= 3218  평균오차율=21.53%
              변수  zone1(≤6%)  zone2(6~8%)  zone3(8~10%,타겟)  zone4(>10%)  z_score(1vs3)
   LDAPS 10m U바람       0.566        2.544            3.076        3.862          0.554
     GFS 10m U바람       0.572        1.632            1.838        2.161          0.554
  LDAPS 50m U최솟값       0.356        3.308            4.020        5.191          0.543
  LDAPS 50m U최댓값       1.407        4.305            5.033        6.233          0.534
     GFS 80m U바람       0.969        2.493            2.840        3.326          0.503
GFS 100m U바람(허브)       1.021        2.610            2.982        3.497          0.497
  GFS 850hPa U바람       2.915        5.313            6.301        7.471          0.444
        GFS 2m기온     285.041      281.781          280.924      280.293         -0.390
      LDAPS 

In [16]:
# ============================================================
# 완전중복(상관계수 1.000) 제거만 — PCA 없이 순수하게 이것만 확인
# ============================================================
DROP_DUPLICATE_FEATS = [
    "gfs_heightAboveGround_80_u",       # gfs_heightAboveGround_100_100u 와 1.000
    "gfs_heightAboveGround_80_v",       # gfs_heightAboveGround_100_100v 와 1.000
    "ldaps_surface_0_avg_lsprate",      # ldaps_surface_0_ncpcp 와 1.000
    "ldaps_surface_0_lssrate",          # ldaps_surface_0_snol 와 1.000
]

FEATS_AFTER_DEDUP = [c for c in RAW_FEATS if c not in DROP_DUPLICATE_FEATS]
print(f"중복제거 후 피처 수: {len(FEATS_AFTER_DEDUP)} (원래 {len(RAW_FEATS)})")

train_datasets_dedup = {g: df.drop(columns=DROP_DUPLICATE_FEATS, errors="ignore") for g, df in train_datasets.items()}
test_datasets_dedup = {g: df.drop(columns=DROP_DUPLICATE_FEATS, errors="ignore") for g, df in test_datasets.items()}


# ============================================================
# 완전중복 제거 버전으로 baseline과 동일 조건 재평가
# ============================================================
dedup_only_results = []
for g in TARGET_COLS:
    df = train_datasets_dedup[g]
    tr = df[(df["forecast_kst_dtm"] >= TRAIN_PERIOD_START[g]) & (df["forecast_kst_dtm"] < CORR_VAL_START)]
    va = df[df["forecast_kst_dtm"] >= CORR_VAL_START].copy()
    capacity = CAPACITY_KWH[g]
    y_va = va["target"].to_numpy(dtype=float)

    for name in TOP2_MODELS[g]:
        model = make_model(name)
        model.fit(tr[FEATS_AFTER_DEDUP], tr["target"])
        pred = np.clip(model.predict(va[FEATS_AFTER_DEDUP]), 0, capacity)
        score, nmae, ficr = group_score(y_va, pred, capacity)
        print(f"[{g}] {name:15s} (중복제거만) -> score={score:.4f} nmae={nmae:.4f} ficr={ficr:.4f}")
        dedup_only_results.append({"group": g, "model": name, "score": score, "nmae": nmae, "ficr": ficr})
    print()

dedup_results_df = pd.DataFrame(dedup_only_results)
print("=" * 70)
print("중복제거만 적용 vs baseline 비교")
print("=" * 70)
compare_dedup = dedup_results_df.merge(
    baseline_summary_df[["group", "model", "score"]].rename(columns={"score": "baseline_score"}),
    on=["group", "model"]
)
compare_dedup["score_diff"] = compare_dedup["score"] - compare_dedup["baseline_score"]
print(compare_dedup[["group", "model", "baseline_score", "score", "score_diff"]].to_string(index=False))

중복제거 후 피처 수: 65 (원래 69)
[kpx_group_1] LightGBM        (중복제거만) -> score=0.6143 nmae=0.1267 ficr=0.3553
[kpx_group_1] HistGB          (중복제거만) -> score=0.6136 nmae=0.1276 ficr=0.3547

[kpx_group_2] HistGB          (중복제거만) -> score=0.6461 nmae=0.1244 ficr=0.4167
[kpx_group_2] LightGBM        (중복제거만) -> score=0.6431 nmae=0.1253 ficr=0.4116

[kpx_group_3] MLP             (중복제거만) -> score=0.5597 nmae=0.1550 ficr=0.2744
[kpx_group_3] RandomForest    (중복제거만) -> score=0.5527 nmae=0.1515 ficr=0.2570

중복제거만 적용 vs baseline 비교
      group        model  baseline_score    score  score_diff
kpx_group_1     LightGBM        0.611286 0.614300    0.003014
kpx_group_1       HistGB        0.619464 0.613561   -0.005902
kpx_group_2       HistGB        0.643940 0.646138    0.002197
kpx_group_2     LightGBM        0.649180 0.643132   -0.006048
kpx_group_3          MLP        0.553396 0.559708    0.006312
kpx_group_3 RandomForest        0.550331 0.552732    0.002401


In [23]:
# ============================================================
# SCADA 전체 변수(풍속+풍향, 터빈별 개별) — baseline과 동일한 시간분할로 검증
# SCADA 기간 확인 완료: 2022~2025(vestas), 2023~2025(unison) - val(2024) 커버됨
# ============================================================

GROUP_SCADA_TURBINES = {
    "kpx_group_1": ("vestas", list(range(1, 7))),
    "kpx_group_2": ("vestas", list(range(7, 13))),
    "kpx_group_3": ("unison", list(range(1, 6))),
}

def build_scada_full_features(group, scada_vestas, scada_unison):
    maker, turbines = GROUP_SCADA_TURBINES[group]
    src = scada_vestas if maker == "vestas" else scada_unison

    ws_cols = [f"{maker}_wtg{t:02d}_ws" for t in turbines]
    wd_cols = [f"{maker}_wtg{t:02d}_wd" for t in turbines]

    df = src[["kst_dtm"] + ws_cols + wd_cols].copy()
    for c in wd_cols:
        df[f"{c}_sin"] = np.sin(np.radians(df[c]))
        df[f"{c}_cos"] = np.cos(np.radians(df[c]))
    df = df.drop(columns=wd_cols)

    df["_hour"] = df["kst_dtm"].dt.floor("h")
    df = df.drop(columns=["kst_dtm"])
    hourly = df.groupby("_hour").mean().reset_index()

    return hourly.rename(columns={"_hour": "forecast_kst_dtm"})


scada_upper_bound_results = []

for g in TARGET_COLS:
    scada_full = build_scada_full_features(g, scada_vestas, scada_unison)

    df = train_datasets[g][["forecast_kst_dtm", "target", "hour_sin", "hour_cos", "month_sin", "month_cos"]].merge(
        scada_full, on="forecast_kst_dtm", how="inner"
    )

    tr = df[(df["forecast_kst_dtm"] >= TRAIN_PERIOD_START[g]) & (df["forecast_kst_dtm"] < CORR_VAL_START)]
    va = df[df["forecast_kst_dtm"] >= CORR_VAL_START]

    capacity = CAPACITY_KWH[g]
    feats = [c for c in df.columns if c not in ("forecast_kst_dtm", "target")]

    print(f"[{g}] n_feats={len(feats)}, n_train={len(tr)}, n_val={len(va)}")

    model = LGBMRegressor(n_estimators=300, max_depth=-1, num_leaves=63, learning_rate=0.05, random_state=42, verbose=-1)
    model.fit(tr[feats], tr["target"])
    pred = np.clip(model.predict(va[feats]), 0, capacity)

    y_va = va["target"].to_numpy(dtype=float)
    score, nmae, ficr = group_score(y_va, pred, capacity)

    print(f"  -> score={score:.4f} nmae={nmae:.4f} ficr={ficr:.4f}")
    scada_upper_bound_results.append({"group": g, "score": score, "nmae": nmae, "ficr": ficr})

print()
print("=" * 70)
print("SCADA 전체(ws+wd) 상한선 vs 현재 baseline(예보풍속) — 동일 기간 비교")
print("=" * 70)
for r in scada_upper_bound_results:
    g = r["group"]
    baseline_best = baseline_summary_df[baseline_summary_df["group"] == g]["score"].max()
    print(f"[{g}] SCADA상한선={r['score']:.4f}  vs  baseline(최고)={baseline_best:.4f}  격차={r['score']-baseline_best:+.4f}")


# ============================================================
# SCADA 전체(ws+wd) 모델의 Feature Importance 확인
# ============================================================
scada_full_importance_models = {}

for g in TARGET_COLS:
    scada_full = build_scada_full_features(g, scada_vestas, scada_unison)

    df = train_datasets[g][["forecast_kst_dtm", "target", "hour_sin", "hour_cos", "month_sin", "month_cos"]].merge(
        scada_full, on="forecast_kst_dtm", how="inner"
    )

    tr = df[(df["forecast_kst_dtm"] >= TRAIN_PERIOD_START[g]) & (df["forecast_kst_dtm"] < CORR_VAL_START)]
    feats = [c for c in df.columns if c not in ("forecast_kst_dtm", "target")]

    model = LGBMRegressor(n_estimators=300, max_depth=-1, num_leaves=63, learning_rate=0.05, random_state=42, verbose=-1)
    model.fit(tr[feats], tr["target"])
    scada_full_importance_models[g] = (model, feats)

    imp_df = pd.DataFrame({
        "변수": feats,
        "importance": model.feature_importances_,
    }).sort_values("importance", ascending=False)
    imp_df["importance_pct"] = (imp_df["importance"] / imp_df["importance"].sum() * 100).round(2)

    print(f"===== {g} — SCADA기반 모델 Feature Importance Top 15 =====")
    print(imp_df[["변수", "importance_pct"]].head(15).to_string(index=False))

    ws_mask = imp_df["변수"].str.contains("_ws")
    wd_mask = imp_df["변수"].str.contains("_wd_")
    ws_total = imp_df.loc[ws_mask, "importance_pct"].sum()
    wd_total = imp_df.loc[wd_mask, "importance_pct"].sum()
    print(f"  >>> 풍속(ws) 합계: {ws_total:.1f}%  |  풍향(wd) 합계: {wd_total:.1f}%")
    print()


# ============================================================
# Ablation: 풍속(ws)만 vs 풍향(wd)만 vs 둘 다 — 실제 score로 검증
# ============================================================

def build_scada_ws_only(group, scada_vestas, scada_unison):
    """풍속만 (터빈별)"""
    maker, turbines = GROUP_SCADA_TURBINES[group]
    src = scada_vestas if maker == "vestas" else scada_unison
    ws_cols = [f"{maker}_wtg{t:02d}_ws" for t in turbines]

    df = src[["kst_dtm"] + ws_cols].copy()
    df["_hour"] = df["kst_dtm"].dt.floor("h")
    df = df.drop(columns=["kst_dtm"])
    hourly = df.groupby("_hour").mean().reset_index()
    return hourly.rename(columns={"_hour": "forecast_kst_dtm"})


def build_scada_wd_only(group, scada_vestas, scada_unison):
    """풍향만 (sin/cos 변환, 터빈별)"""
    maker, turbines = GROUP_SCADA_TURBINES[group]
    src = scada_vestas if maker == "vestas" else scada_unison
    wd_cols = [f"{maker}_wtg{t:02d}_wd" for t in turbines]

    df = src[["kst_dtm"] + wd_cols].copy()
    for c in wd_cols:
        df[f"{c}_sin"] = np.sin(np.radians(df[c]))
        df[f"{c}_cos"] = np.cos(np.radians(df[c]))
    df = df.drop(columns=wd_cols)
    df["_hour"] = df["kst_dtm"].dt.floor("h")
    df = df.drop(columns=["kst_dtm"])
    hourly = df.groupby("_hour").mean().reset_index()
    return hourly.rename(columns={"_hour": "forecast_kst_dtm"})


SCADA_VARIANTS = {
    "풍속(ws)만": build_scada_ws_only,
    "풍향(wd)만": build_scada_wd_only,
    "풍속+풍향(전체)": build_scada_full_features,
}

ablation_results = []

for g in TARGET_COLS:
    print(f"===== {g} =====")
    for variant_name, build_func in SCADA_VARIANTS.items():
        scada_variant = build_func(g, scada_vestas, scada_unison)

        df = train_datasets[g][["forecast_kst_dtm", "target", "hour_sin", "hour_cos", "month_sin", "month_cos"]].merge(
            scada_variant, on="forecast_kst_dtm", how="inner"
        )
        tr = df[(df["forecast_kst_dtm"] >= TRAIN_PERIOD_START[g]) & (df["forecast_kst_dtm"] < CORR_VAL_START)]
        va = df[df["forecast_kst_dtm"] >= CORR_VAL_START].copy()

        capacity = CAPACITY_KWH[g]
        feats = [c for c in df.columns if c not in ("forecast_kst_dtm", "target")]

        model = LGBMRegressor(n_estimators=300, max_depth=-1, num_leaves=63, learning_rate=0.05, random_state=42, verbose=-1)
        model.fit(tr[feats], tr["target"])
        pred = np.clip(model.predict(va[feats]), 0, capacity)

        y_va = va["target"].to_numpy(dtype=float)
        score, nmae, ficr = group_score(y_va, pred, capacity)

        print(f"  {variant_name:15s} (n_feats={len(feats):2d}) -> score={score:.4f} nmae={nmae:.4f} ficr={ficr:.4f}")
        ablation_results.append({"group": g, "variant": variant_name, "n_feats": len(feats),
                                  "score": score, "nmae": nmae, "ficr": ficr})
    print()

ablation_df = pd.DataFrame(ablation_results)
print("=" * 70)
print("풍속 vs 풍향 Ablation 요약")
print("=" * 70)
print(ablation_df.pivot(index="variant", columns="group", values="score").round(4))

[kpx_group_1] n_feats=22, n_train=8757, n_val=8779
  -> score=0.7042 nmae=0.0900 ficr=0.4984
[kpx_group_2] n_feats=22, n_train=17422, n_val=8779
  -> score=0.7389 nmae=0.0854 ficr=0.5633
[kpx_group_3] n_feats=19, n_train=8759, n_val=8779
  -> score=0.6447 nmae=0.1049 ficr=0.3944

SCADA 전체(ws+wd) 상한선 vs 현재 baseline(예보풍속) — 동일 기간 비교
[kpx_group_1] SCADA상한선=0.7042  vs  baseline(최고)=0.6195  격차=+0.0847
[kpx_group_2] SCADA상한선=0.7389  vs  baseline(최고)=0.6492  격차=+0.0897
[kpx_group_3] SCADA상한선=0.6447  vs  baseline(최고)=0.5534  격차=+0.0913
===== kpx_group_1 — SCADA기반 모델 Feature Importance Top 15 =====
                 변수  importance_pct
    vestas_wtg04_ws            7.49
    vestas_wtg05_ws            7.45
    vestas_wtg01_ws            6.29
    vestas_wtg06_ws            6.24
    vestas_wtg03_ws            5.99
    vestas_wtg02_ws            5.93
vestas_wtg03_wd_sin            4.95
vestas_wtg05_wd_sin            4.64
vestas_wtg04_wd_sin            4.52
           hour_sin            4.49
vestas_

### SCADA 데이터 원본을 쓰니 훨씬 성능이 좋아짐 --> 풍속을 어떻게든 잘 맞추는게 중요한 과제이지 않을까??


In [24]:
# ============================================================
# 0. SCADA 로드
# ============================================================
GROUP_SCADA_TURBINES = {
    "kpx_group_1": ("vestas", list(range(1, 7))),
    "kpx_group_2": ("vestas", list(range(7, 13))),
    "kpx_group_3": ("unison", list(range(1, 6))),
}

scada_vestas = pd.read_csv(TRAIN_DIR / "scada_vestas_train.csv", encoding="utf-8-sig")
scada_vestas["kst_dtm"] = pd.to_datetime(scada_vestas["kst_dtm"])

scada_unison = pd.read_csv(TRAIN_DIR / "scada_unison_train.csv", encoding="utf-8-sig")
scada_unison["kst_dtm"] = pd.to_datetime(scada_unison["kst_dtm"])

print("scada_vestas 기간:", scada_vestas["kst_dtm"].min(), "~", scada_vestas["kst_dtm"].max())
print("scada_unison 기간:", scada_unison["kst_dtm"].min(), "~", scada_unison["kst_dtm"].max())
print()


# ============================================================
# 1. 보정모델 타겟(터빈 평균 풍속) 생성 함수
# ============================================================
def build_scada_wind_target_avg(group, scada_vestas, scada_unison):
    """터빈 평균 풍속 (보정모델 타겟용)"""
    maker, turbines = GROUP_SCADA_TURBINES[group]
    src = scada_vestas if maker == "vestas" else scada_unison
    ws_cols = [f"{maker}_wtg{t:02d}_ws" for t in turbines]

    turbine_mean = src[ws_cols].mean(axis=1)
    tmp = pd.DataFrame({"kst_dtm": src["kst_dtm"], "scada_ws": turbine_mean})
    tmp["_hour"] = tmp["kst_dtm"].dt.floor("h")
    tmp = tmp.drop(columns=["kst_dtm"])
    hourly = tmp.groupby("_hour").mean().reset_index()
    return hourly.rename(columns={"_hour": "forecast_kst_dtm"})


# ============================================================
# 2. NaN 진단 (미리 확인)
# ============================================================
print("=" * 50)
print("타겟(scada_ws) 결측 진단")
print("=" * 50)
for g in TARGET_COLS:
    scada_target = build_scada_wind_target_avg(g, scada_vestas, scada_unison)
    n_nan = scada_target["scada_ws"].isna().sum()
    print(f"[{g}] 전체 {len(scada_target)}행 중 결측 {n_nan}개")
print()


# ============================================================
# 3. 보정모델 후보 5개 정의
# ============================================================
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error
from lightgbm import LGBMRegressor

def build_corr_models():
    return {
        "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
        "RandomForest": RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42, n_jobs=-1),
        "HistGB": HistGradientBoostingRegressor(max_iter=300, max_depth=8, random_state=42),
        "LightGBM": LGBMRegressor(n_estimators=300, max_depth=8, learning_rate=0.05, random_state=42, verbose=-1),
        "MLP": make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(32, 16), early_stopping=True, random_state=42, max_iter=2000)),
    }


# ============================================================
# 4. 그룹별 RAW_FEATS(원본 전체) 기준으로 5개 모델 MAE 비교
#    (merge 후 target NaN은 반드시 dropna로 제거 - 데이터 시작시각이 소스마다 다를 수 있음)
# ============================================================
correction_model_comparison = []

for g in TARGET_COLS:
    scada_target = build_scada_wind_target_avg(g, scada_vestas, scada_unison)

    df = train_datasets[g][["forecast_kst_dtm"] + RAW_FEATS].merge(
        scada_target, on="forecast_kst_dtm", how="inner"
    )
    df = df.dropna(subset=["scada_ws"])  # NaN 방어

    tr = df[(df["forecast_kst_dtm"] >= TRAIN_PERIOD_START[g]) & (df["forecast_kst_dtm"] < CORR_VAL_START)]
    va = df[df["forecast_kst_dtm"] >= CORR_VAL_START].copy()

    print(f"===== {g} (n_feats={len(RAW_FEATS)}, n_train={len(tr)}, n_val={len(va)}) =====")
    for name, model in build_corr_models().items():
        model.fit(tr[RAW_FEATS], tr["scada_ws"])
        pred = model.predict(va[RAW_FEATS])
        mae = mean_absolute_error(va["scada_ws"], pred)
        print(f"  {name:15s} MAE={mae:.3f}")
        correction_model_comparison.append({"group": g, "model": name, "mae": mae})
    print()

corr_comp_df = pd.DataFrame(correction_model_comparison)
print("=" * 50)
print("보정모델 MAE 비교 (낮을수록 좋음)")
print("=" * 50)
print(corr_comp_df.pivot(index="model", columns="group", values="mae").round(3))

scada_vestas 기간: 2022-01-01 01:00:00 ~ 2025-01-01 00:00:00
scada_unison 기간: 2023-01-01 00:10:00 ~ 2025-01-01 00:00:00

타겟(scada_ws) 결측 진단
[kpx_group_1] 전체 26304행 중 결측 0개
[kpx_group_2] 전체 26304행 중 결측 0개
[kpx_group_3] 전체 17545행 중 결측 8개

===== kpx_group_1 (n_feats=69, n_train=8757, n_val=8779) =====
  Ridge           MAE=1.340
  RandomForest    MAE=1.118
  HistGB          MAE=1.113
  LightGBM        MAE=1.105
  MLP             MAE=1.314

===== kpx_group_2 (n_feats=69, n_train=17422, n_val=8779) =====
  Ridge           MAE=1.478
  RandomForest    MAE=1.226
  HistGB          MAE=1.196
  LightGBM        MAE=1.178
  MLP             MAE=1.303

===== kpx_group_3 (n_feats=69, n_train=8757, n_val=8778) =====
  Ridge           MAE=1.409
  RandomForest    MAE=1.172
  HistGB          MAE=1.164
  LightGBM        MAE=1.151
  MLP             MAE=1.330

보정모델 MAE 비교 (낮을수록 좋음)
group         kpx_group_1  kpx_group_2  kpx_group_3
model                                              
HistGB              1.113 

In [25]:
# ============================================================
# 1. LightGBM 보정모델 학습 (그룹별) + corrected_ws 생성
# ============================================================
wind_correction_models = {}

for g in TARGET_COLS:
    scada_target = build_scada_wind_target_avg(g, scada_vestas, scada_unison)

    df_corr = train_datasets[g][["forecast_kst_dtm"] + RAW_FEATS].merge(
        scada_target, on="forecast_kst_dtm", how="inner"
    )
    df_corr = df_corr.dropna(subset=["scada_ws"])

    tr_corr = df_corr[(df_corr["forecast_kst_dtm"] >= TRAIN_PERIOD_START[g]) & (df_corr["forecast_kst_dtm"] < CORR_VAL_START)]

    model = LGBMRegressor(n_estimators=300, max_depth=8, learning_rate=0.05, random_state=42, verbose=-1)
    model.fit(tr_corr[RAW_FEATS], tr_corr["scada_ws"])
    wind_correction_models[g] = model

    print(f"[{g}] 보정모델 학습 완료 (n_train={len(tr_corr)})")

print()

# ============================================================
# 2. train/test 전체에 corrected_ws 컬럼 추가
# ============================================================
for g in TARGET_COLS:
    model = wind_correction_models[g]
    train_datasets[g]["corrected_ws"] = model.predict(train_datasets[g][RAW_FEATS])
    test_datasets[g]["corrected_ws"] = model.predict(test_datasets[g][RAW_FEATS])
    print(f"[{g}] corrected_ws 추가 완료 - train 평균={train_datasets[g]['corrected_ws'].mean():.2f}, "
          f"test 평균={test_datasets[g]['corrected_ws'].mean():.2f}")

print()

# ============================================================
# 3. corrected_ws 포함한 새 피처셋으로 최종 발전량 모델(top2) 재학습 + 평가
# ============================================================
FEATS_WITH_CORRECTION = RAW_FEATS + ["corrected_ws"]

corrected_results = []

for g in TARGET_COLS:
    df = train_datasets[g]
    tr = df[(df["forecast_kst_dtm"] >= TRAIN_PERIOD_START[g]) & (df["forecast_kst_dtm"] < CORR_VAL_START)]
    va = df[df["forecast_kst_dtm"] >= CORR_VAL_START].copy()
    capacity = CAPACITY_KWH[g]
    y_va = va["target"].to_numpy(dtype=float)

    for name in TOP2_MODELS[g]:
        model = make_model(name)
        model.fit(tr[FEATS_WITH_CORRECTION], tr["target"])
        pred = np.clip(model.predict(va[FEATS_WITH_CORRECTION]), 0, capacity)
        score, nmae, ficr = group_score(y_va, pred, capacity)

        print(f"[{g}] {name:15s} (corrected_ws 추가) -> score={score:.4f} nmae={nmae:.4f} ficr={ficr:.4f}")
        corrected_results.append({"group": g, "model": name, "score": score, "nmae": nmae, "ficr": ficr})
    print()

corrected_results_df = pd.DataFrame(corrected_results)

# ============================================================
# 4. baseline과 비교
# ============================================================
print("=" * 70)
print("corrected_ws 추가 vs baseline 비교")
print("=" * 70)
compare_corrected = corrected_results_df.merge(
    baseline_summary_df[["group", "model", "score"]].rename(columns={"score": "baseline_score"}),
    on=["group", "model"]
)
compare_corrected["score_diff"] = compare_corrected["score"] - compare_corrected["baseline_score"]
print(compare_corrected[["group", "model", "baseline_score", "score", "score_diff"]].to_string(index=False))

[kpx_group_1] 보정모델 학습 완료 (n_train=8757)
[kpx_group_2] 보정모델 학습 완료 (n_train=17422)
[kpx_group_3] 보정모델 학습 완료 (n_train=8757)

[kpx_group_1] corrected_ws 추가 완료 - train 평균=6.88, test 평균=7.32
[kpx_group_2] corrected_ws 추가 완료 - train 평균=7.23, test 평균=7.68
[kpx_group_3] corrected_ws 추가 완료 - train 평균=5.85, test 평균=6.27

[kpx_group_1] LightGBM        (corrected_ws 추가) -> score=0.6157 nmae=0.1291 ficr=0.3605
[kpx_group_1] HistGB          (corrected_ws 추가) -> score=0.6171 nmae=0.1288 ficr=0.3630

[kpx_group_2] HistGB          (corrected_ws 추가) -> score=0.6435 nmae=0.1286 ficr=0.4156
[kpx_group_2] LightGBM        (corrected_ws 추가) -> score=0.6486 nmae=0.1267 ficr=0.4239

[kpx_group_3] MLP             (corrected_ws 추가) -> score=0.5604 nmae=0.1534 ficr=0.2743
[kpx_group_3] RandomForest    (corrected_ws 추가) -> score=0.5653 nmae=0.1462 ficr=0.2768

corrected_ws 추가 vs baseline 비교
      group        model  baseline_score    score  score_diff
kpx_group_1     LightGBM        0.611286 0.615739    0.004452
kp

In [26]:
# 보정된 풍속 vs 실제 SCADA의 변동성(표준편차) 비교
for g in TARGET_COLS:
    scada_target = build_scada_wind_target_avg(g, scada_vestas, scada_unison)
    va_check = train_datasets[g][train_datasets[g]["forecast_kst_dtm"] >= CORR_VAL_START]
    va_check = va_check.merge(scada_target, on="forecast_kst_dtm", how="inner")

    print(f"[{g}] corrected_ws 표준편차: {va_check['corrected_ws'].std():.3f}  "
          f"vs 실제 SCADA 표준편차: {va_check['scada_ws'].std():.3f}")

[kpx_group_1] corrected_ws 표준편차: 3.129  vs 실제 SCADA 표준편차: 3.357
[kpx_group_2] corrected_ws 표준편차: 3.552  vs 실제 SCADA 표준편차: 3.814
[kpx_group_3] corrected_ws 표준편차: 3.245  vs 실제 SCADA 표준편차: 3.635


In [27]:
# ============================================================
# 분산 복원(rescaling) 적용한 corrected_ws_v2 생성
# ============================================================

scale_factors = {}

for g in TARGET_COLS:
    scada_target = build_scada_wind_target_avg(g, scada_vestas, scada_unison)

    # train 기간에서 scale factor 계산 (val로 계산하면 leakage)
    df_check = train_datasets[g][["forecast_kst_dtm", "corrected_ws"]].merge(
        scada_target, on="forecast_kst_dtm", how="inner"
    )
    tr_check = df_check[(df_check["forecast_kst_dtm"] >= TRAIN_PERIOD_START[g]) & (df_check["forecast_kst_dtm"] < CORR_VAL_START)]

    corrected_std = tr_check["corrected_ws"].std()
    scada_std = tr_check["scada_ws"].std()
    scale_factor = scada_std / corrected_std
    scale_factors[g] = scale_factor

    print(f"[{g}] train 기준 scale_factor={scale_factor:.4f} (corrected_std={corrected_std:.3f}, scada_std={scada_std:.3f})")

print()

# train/test 둘 다에 rescaling 적용 (평균은 유지, 편차만 확대)
for g in TARGET_COLS:
    for ds in [train_datasets[g], test_datasets[g]]:
        mean_val = ds["corrected_ws"].mean()
        ds["corrected_ws_v2"] = mean_val + (ds["corrected_ws"] - mean_val) * scale_factors[g]

    print(f"[{g}] corrected_ws_v2 생성 완료 - std={train_datasets[g]['corrected_ws_v2'].std():.3f}")


# ============================================================
# corrected_ws_v2로 최종 발전량 모델 재평가
# ============================================================
FEATS_WITH_CORRECTION_V2 = RAW_FEATS + ["corrected_ws_v2"]

corrected_v2_results = []

for g in TARGET_COLS:
    df = train_datasets[g]
    tr = df[(df["forecast_kst_dtm"] >= TRAIN_PERIOD_START[g]) & (df["forecast_kst_dtm"] < CORR_VAL_START)]
    va = df[df["forecast_kst_dtm"] >= CORR_VAL_START].copy()
    capacity = CAPACITY_KWH[g]
    y_va = va["target"].to_numpy(dtype=float)

    for name in TOP2_MODELS[g]:
        model = make_model(name)
        model.fit(tr[FEATS_WITH_CORRECTION_V2], tr["target"])
        pred = np.clip(model.predict(va[FEATS_WITH_CORRECTION_V2]), 0, capacity)
        score, nmae, ficr = group_score(y_va, pred, capacity)
        print(f"[{g}] {name:15s} (corrected_ws_v2, 분산복원) -> score={score:.4f} nmae={nmae:.4f} ficr={ficr:.4f}")
        corrected_v2_results.append({"group": g, "model": name, "score": score, "nmae": nmae, "ficr": ficr})
    print()

corrected_v2_df = pd.DataFrame(corrected_v2_results)
print("=" * 70)
print("corrected_ws_v2(분산복원) vs baseline 비교")
print("=" * 70)
compare_v2 = corrected_v2_df.merge(
    baseline_summary_df[["group", "model", "score"]].rename(columns={"score": "baseline_score"}),
    on=["group", "model"]
)
compare_v2["score_diff"] = compare_v2["score"] - compare_v2["baseline_score"]
print(compare_v2[["group", "model", "baseline_score", "score", "score_diff"]].to_string(index=False))

[kpx_group_1] train 기준 scale_factor=1.0617 (corrected_std=3.190, scada_std=3.387)
[kpx_group_2] train 기준 scale_factor=1.0768 (corrected_std=3.574, scada_std=3.848)
[kpx_group_3] train 기준 scale_factor=1.0614 (corrected_std=3.368, scada_std=3.575)

[kpx_group_1] corrected_ws_v2 생성 완료 - std=3.318
[kpx_group_2] corrected_ws_v2 생성 완료 - std=3.842
[kpx_group_3] corrected_ws_v2 생성 완료 - std=3.510
[kpx_group_1] LightGBM        (corrected_ws_v2, 분산복원) -> score=0.6157 nmae=0.1291 ficr=0.3605
[kpx_group_1] HistGB          (corrected_ws_v2, 분산복원) -> score=0.6171 nmae=0.1288 ficr=0.3630

[kpx_group_2] HistGB          (corrected_ws_v2, 분산복원) -> score=0.6435 nmae=0.1286 ficr=0.4156
[kpx_group_2] LightGBM        (corrected_ws_v2, 분산복원) -> score=0.6486 nmae=0.1267 ficr=0.4239

[kpx_group_3] MLP             (corrected_ws_v2, 분산복원) -> score=0.5604 nmae=0.1534 ficr=0.2743
[kpx_group_3] RandomForest    (corrected_ws_v2, 분산복원) -> score=0.5653 nmae=0.1462 ficr=0.2768

corrected_ws_v2(분산복원) vs baseline 비교
     

In [28]:
for g in TARGET_COLS:
    scada_target = build_scada_wind_target_avg(g, scada_vestas, scada_unison)
    va_check = train_datasets[g][train_datasets[g]["forecast_kst_dtm"] >= CORR_VAL_START].copy()
    va_check = va_check.merge(scada_target, on="forecast_kst_dtm", how="inner")

    va_check["ws_bin"] = pd.cut(va_check["scada_ws"], bins=[0, 3, 6, 9, 12, 15, 100],
                                  labels=["0-3", "3-6", "6-9", "9-12", "12-15", "15+"])
    va_check["signed_error"] = va_check["corrected_ws"] - va_check["scada_ws"]
    va_check["abs_error"] = va_check["signed_error"].abs()

    summary = va_check.groupby("ws_bin", observed=True).agg(
        n=("scada_ws", "size"),
        scada_mean=("scada_ws", "mean"),
        corrected_mean=("corrected_ws", "mean"),
        signed_error_mean=("signed_error", "mean"),
        abs_error_mean=("abs_error", "mean"),
    )
    print(f"===== {g} =====")
    print(summary.round(3))
    print()

===== kpx_group_1 =====
           n  scada_mean  corrected_mean  signed_error_mean  abs_error_mean
ws_bin                                                                     
0-3     1022       2.301           3.384              1.083           1.144
3-6     3178       4.465           4.577              0.112           0.865
6-9     2353       7.412           7.420              0.008           1.154
9-12    1508      10.257           9.944             -0.314           1.167
12-15    575      13.238          12.035             -1.202           1.661
15+      141      16.156          13.974             -2.182           2.380

===== kpx_group_2 =====
           n  scada_mean  corrected_mean  signed_error_mean  abs_error_mean
ws_bin                                                                     
0-3     1005       2.317           3.424              1.107           1.133
3-6     3147       4.433           4.507              0.074           0.851
6-9     2121       7.430           7.53

In [32]:
# ============================================================
# 0. BEST_MODEL_PER_GROUP 재생성 (baseline_summary_df 기준)
# ============================================================
BEST_MODEL_PER_GROUP = (
    baseline_summary_df.loc[baseline_summary_df.groupby("group")["score"].idxmax()]
    .set_index("group")["model"].to_dict()
)
print("그룹별 최고 모델:", BEST_MODEL_PER_GROUP)
print()

# ============================================================
# 보정모델 오차(SCADA 기준)와 발전량 예측 오차(target 기준)가
# 실제로 같은 시간대에서 같이 커지는지 대조 — 그룹별 top2 모델 둘 다
# ============================================================

for g in TARGET_COLS:
    scada_target = build_scada_wind_target_avg(g, scada_vestas, scada_unison)
    va_wind = train_datasets[g][train_datasets[g]["forecast_kst_dtm"] >= CORR_VAL_START][
        ["forecast_kst_dtm", "corrected_ws"]
    ].merge(scada_target, on="forecast_kst_dtm", how="inner")
    va_wind["wind_abs_error"] = (va_wind["corrected_ws"] - va_wind["scada_ws"]).abs()

    for model_name in TOP2_MODELS[g]:
        va_power = baseline_val_results[(g, model_name)].copy()
        va_power["power_error_rate"] = np.abs(va_power["pred"] - va_power["target"]) / CAPACITY_KWH[g]

        merged = va_power[["forecast_kst_dtm", "power_error_rate"]].merge(
            va_wind[["forecast_kst_dtm", "wind_abs_error", "scada_ws"]], on="forecast_kst_dtm", how="inner"
        )
        merged["ws_bin"] = pd.cut(merged["scada_ws"], bins=[0, 3, 6, 9, 12, 15, 100],
                                    labels=["0-3", "3-6", "6-9", "9-12", "12-15", "15+"])

        print(f"===== {g} / {model_name} =====")
        summary = merged.groupby("ws_bin", observed=True).agg(
            n=("scada_ws", "size"),
            wind_abs_error_mean=("wind_abs_error", "mean"),
            power_error_rate_mean=("power_error_rate", "mean"),
        )
        print(summary.round(4))

        corr = merged[["wind_abs_error", "power_error_rate"]].corr().iloc[0, 1]
        print(f"  풍속오차 vs 발전량오차율 상관계수: {corr:.3f}")
        print()


그룹별 최고 모델: {'kpx_group_1': 'HistGB', 'kpx_group_2': 'LightGBM', 'kpx_group_3': 'MLP'}

===== kpx_group_1 / LightGBM =====
           n  wind_abs_error_mean  power_error_rate_mean
ws_bin                                                  
0-3     1022               1.1438                 0.0353
3-6     3178               0.8652                 0.0633
6-9     2353               1.1540                 0.1381
9-12    1508               1.1673                 0.1582
12-15    575               1.6613                 0.1531
15+      141               2.3796                 0.1101
  풍속오차 vs 발전량오차율 상관계수: 0.368

===== kpx_group_1 / HistGB =====
           n  wind_abs_error_mean  power_error_rate_mean
ws_bin                                                  
0-3     1022               1.1438                 0.0375
3-6     3178               0.8652                 0.0645
6-9     2353               1.1540                 0.1403
9-12    1508               1.1673                 0.1596
12-15    575     

In [33]:
# ============================================================
# 그룹별로 다른 구간에 가중치 부여 (보정모델 재학습)
# 그룹1·2: 9-12 구간 집중 (발전량오차율이 가장 높았던 지점)
# 그룹3: 15+ 구간 집중 (풍속 셀수록 계속 나빠지는 패턴)
# ============================================================

def make_group_specific_weight(group, scada_ws):
    weight = np.ones(len(scada_ws))
    if group in ("kpx_group_1", "kpx_group_2"):
        # 9~12 구간에 가중치 5배
        weight = np.where((scada_ws >= 9) & (scada_ws < 12), 5.0, weight)
        # 12~15, 15+도 어느 정도는 챙기되 조금 약하게
        weight = np.where(scada_ws >= 12, 3.0, weight)
    else:  # kpx_group_3
        # 15+ 구간에 강하게 집중, 9~15도 단계적으로
        weight = np.where((scada_ws >= 9) & (scada_ws < 12), 2.0, weight)
        weight = np.where((scada_ws >= 12) & (scada_ws < 15), 4.0, weight)
        weight = np.where(scada_ws >= 15, 6.0, weight)
    return weight


wind_correction_models_v3 = {}

for g in TARGET_COLS:
    scada_target = build_scada_wind_target_avg(g, scada_vestas, scada_unison)

    df_corr = train_datasets[g][["forecast_kst_dtm"] + RAW_FEATS].merge(
        scada_target, on="forecast_kst_dtm", how="inner"
    )
    df_corr = df_corr.dropna(subset=["scada_ws"])

    tr_corr = df_corr[(df_corr["forecast_kst_dtm"] >= TRAIN_PERIOD_START[g]) & (df_corr["forecast_kst_dtm"] < CORR_VAL_START)]
    va_corr = df_corr[df_corr["forecast_kst_dtm"] >= CORR_VAL_START].copy()

    sw = make_group_specific_weight(g, tr_corr["scada_ws"].to_numpy())

    model = LGBMRegressor(n_estimators=300, max_depth=8, learning_rate=0.05, random_state=42, verbose=-1)
    model.fit(tr_corr[RAW_FEATS], tr_corr["scada_ws"], sample_weight=sw)
    wind_correction_models_v3[g] = model

    va_corr["corrected_ws_v3"] = model.predict(va_corr[RAW_FEATS])
    va_corr["ws_bin"] = pd.cut(va_corr["scada_ws"], bins=[0, 3, 6, 9, 12, 15, 100],
                                 labels=["0-3", "3-6", "6-9", "9-12", "12-15", "15+"])
    va_corr["abs_error"] = (va_corr["corrected_ws_v3"] - va_corr["scada_ws"]).abs()
    va_corr["signed_error"] = va_corr["corrected_ws_v3"] - va_corr["scada_ws"]

    print(f"===== {g} (구간별 가중치 적용) =====")
    summary = va_corr.groupby("ws_bin", observed=True).agg(
        n=("scada_ws", "size"),
        scada_mean=("scada_ws", "mean"),
        corrected_mean=("corrected_ws_v3", "mean"),
        signed_error_mean=("signed_error", "mean"),
        abs_error_mean=("abs_error", "mean"),
    )
    print(summary.round(3))
    print()


# ============================================================
# corrected_ws_v3를 train/test 전체에 반영
# ============================================================
for g in TARGET_COLS:
    model = wind_correction_models_v3[g]
    train_datasets[g]["corrected_ws_v3"] = model.predict(train_datasets[g][RAW_FEATS])
    test_datasets[g]["corrected_ws_v3"] = model.predict(test_datasets[g][RAW_FEATS])


# ============================================================
# corrected_ws_v3 포함해서 최종 발전량 모델 재평가 (top2 둘 다)
# ============================================================
FEATS_WITH_V3 = RAW_FEATS + ["corrected_ws_v3"]

v3_results = []

for g in TARGET_COLS:
    df = train_datasets[g]
    tr = df[(df["forecast_kst_dtm"] >= TRAIN_PERIOD_START[g]) & (df["forecast_kst_dtm"] < CORR_VAL_START)]
    va = df[df["forecast_kst_dtm"] >= CORR_VAL_START].copy()
    capacity = CAPACITY_KWH[g]
    y_va = va["target"].to_numpy(dtype=float)

    for name in TOP2_MODELS[g]:
        model = make_model(name)
        model.fit(tr[FEATS_WITH_V3], tr["target"])
        pred = np.clip(model.predict(va[FEATS_WITH_V3]), 0, capacity)
        score, nmae, ficr = group_score(y_va, pred, capacity)
        print(f"[{g}] {name:15s} (corrected_ws_v3, 구간가중치) -> score={score:.4f} nmae={nmae:.4f} ficr={ficr:.4f}")
        v3_results.append({"group": g, "model": name, "score": score, "nmae": nmae, "ficr": ficr})
    print()

v3_results_df = pd.DataFrame(v3_results)
print("=" * 70)
print("corrected_ws_v3(구간가중치) vs baseline 비교")
print("=" * 70)
compare_v3 = v3_results_df.merge(
    baseline_summary_df[["group", "model", "score"]].rename(columns={"score": "baseline_score"}),
    on=["group", "model"]
)
compare_v3["score_diff"] = compare_v3["score"] - compare_v3["baseline_score"]
print(compare_v3[["group", "model", "baseline_score", "score", "score_diff"]].to_string(index=False))

===== kpx_group_1 (구간별 가중치 적용) =====
           n  scada_mean  corrected_mean  signed_error_mean  abs_error_mean
ws_bin                                                                     
0-3     1022       2.301           3.428              1.127           1.163
3-6     3178       4.465           4.593              0.127           0.886
6-9     2353       7.412           7.710              0.299           1.291
9-12    1508      10.257          10.115             -0.143           1.042
12-15    575      13.238          12.048             -1.190           1.649
15+      141      16.156          14.066             -2.090           2.307

===== kpx_group_2 (구간별 가중치 적용) =====
           n  scada_mean  corrected_mean  signed_error_mean  abs_error_mean
ws_bin                                                                     
0-3     1005       2.317           3.455              1.138           1.160
3-6     3147       4.433           4.522              0.089           0.927
6-9     2121 

In [34]:
# ============================================================
# v1(보정만) vs v3(구간가중치) vs baseline 전체 비교
# ============================================================

# v1 결과는 이미 corrected_results_df 에 저장되어 있음 (이전 실행 결과)
v1_summary = corrected_results_df.rename(columns={"score": "v1_score"})[["group", "model", "v1_score"]]
v3_summary = v3_results_df.rename(columns={"score": "v3_score"})[["group", "model", "v3_score"]]

final_compare = baseline_summary_df[["group", "model", "score"]].rename(columns={"score": "baseline_score"})
final_compare = final_compare.merge(v1_summary, on=["group", "model"], how="left")
final_compare = final_compare.merge(v3_summary, on=["group", "model"], how="left")

final_compare["v1_diff"] = final_compare["v1_score"] - final_compare["baseline_score"]
final_compare["v3_diff"] = final_compare["v3_score"] - final_compare["baseline_score"]

print(final_compare.to_string(index=False))

      group        model  baseline_score  v1_score  v3_score   v1_diff   v3_diff
kpx_group_1     LightGBM        0.611286  0.615739  0.617855  0.004452  0.006569
kpx_group_1       HistGB        0.619464  0.617128  0.617877 -0.002335 -0.001587
kpx_group_2       HistGB        0.643940  0.643516  0.647696 -0.000425  0.003755
kpx_group_2     LightGBM        0.649180  0.648559  0.654190 -0.000621  0.005010
kpx_group_3          MLP        0.553396  0.560431  0.554557  0.007035  0.001161
kpx_group_3 RandomForest        0.550331  0.565297  0.564261  0.014967  0.013930


In [35]:
# train 기간에서도 15+구간을 얼마나 잘 맞추는지 확인
for g in TARGET_COLS:
    scada_target = build_scada_wind_target_avg(g, scada_vestas, scada_unison)
    df_corr = train_datasets[g][["forecast_kst_dtm"] + RAW_FEATS].merge(
        scada_target, on="forecast_kst_dtm", how="inner"
    ).dropna(subset=["scada_ws"])
    tr_corr = df_corr[(df_corr["forecast_kst_dtm"] >= TRAIN_PERIOD_START[g]) & (df_corr["forecast_kst_dtm"] < CORR_VAL_START)]

    model = wind_correction_models_v3[g]
    tr_corr = tr_corr.copy()
    tr_corr["pred"] = model.predict(tr_corr[RAW_FEATS])
    tr_corr["ws_bin"] = pd.cut(tr_corr["scada_ws"], bins=[0,3,6,9,12,15,100], labels=["0-3","3-6","6-9","9-12","12-15","15+"])
    tr_corr["signed_error"] = tr_corr["pred"] - tr_corr["scada_ws"]

    print(f"[{g}] TRAIN 기준 15+구간 signed_error:",
          tr_corr[tr_corr["ws_bin"]=="15+"]["signed_error"].mean(),
          f"(n={len(tr_corr[tr_corr['ws_bin']=='15+'])})")

[kpx_group_1] TRAIN 기준 15+구간 signed_error: -0.5889439428684132 (n=127)
[kpx_group_2] TRAIN 기준 15+구간 signed_error: -0.9423803051537905 (n=639)
[kpx_group_3] TRAIN 기준 15+구간 signed_error: -0.26187289701492017 (n=138)


In [36]:
# ============================================================
# 과적합 억제 버전: 가중치 유지 + 모델 복잡도 낮춤 (정규화 강화)
# ============================================================

def make_group_specific_weight(group, scada_ws):
    weight = np.ones(len(scada_ws))
    if group in ("kpx_group_1", "kpx_group_2"):
        weight = np.where((scada_ws >= 9) & (scada_ws < 12), 5.0, weight)
        weight = np.where(scada_ws >= 12, 3.0, weight)
    else:  # kpx_group_3
        weight = np.where((scada_ws >= 9) & (scada_ws < 12), 2.0, weight)
        weight = np.where((scada_ws >= 12) & (scada_ws < 15), 4.0, weight)
        weight = np.where(scada_ws >= 15, 6.0, weight)
    return weight


wind_correction_models_v4 = {}

for g in TARGET_COLS:
    scada_target = build_scada_wind_target_avg(g, scada_vestas, scada_unison)

    df_corr = train_datasets[g][["forecast_kst_dtm"] + RAW_FEATS].merge(
        scada_target, on="forecast_kst_dtm", how="inner"
    )
    df_corr = df_corr.dropna(subset=["scada_ws"])

    tr_corr = df_corr[(df_corr["forecast_kst_dtm"] >= TRAIN_PERIOD_START[g]) & (df_corr["forecast_kst_dtm"] < CORR_VAL_START)]
    va_corr = df_corr[df_corr["forecast_kst_dtm"] >= CORR_VAL_START].copy()

    sw = make_group_specific_weight(g, tr_corr["scada_ws"].to_numpy())

    model = LGBMRegressor(
        n_estimators=300,
        max_depth=4,
        num_leaves=15,
        min_child_samples=50,
        learning_rate=0.03,
        reg_alpha=1.0,
        reg_lambda=1.0,
        random_state=42,
        verbose=-1,
    )
    model.fit(tr_corr[RAW_FEATS], tr_corr["scada_ws"], sample_weight=sw)
    wind_correction_models_v4[g] = model

    # train 성능
    tr_corr = tr_corr.copy()
    tr_corr["pred"] = model.predict(tr_corr[RAW_FEATS])
    tr_corr["ws_bin"] = pd.cut(tr_corr["scada_ws"], bins=[0, 3, 6, 9, 12, 15, 100],
                                 labels=["0-3", "3-6", "6-9", "9-12", "12-15", "15+"])
    tr_15plus = tr_corr[tr_corr["ws_bin"] == "15+"]
    tr_signed_error = (tr_15plus["pred"] - tr_15plus["scada_ws"]).mean()

    # val 성능
    va_corr["corrected_ws_v4"] = model.predict(va_corr[RAW_FEATS])
    va_corr["ws_bin"] = pd.cut(va_corr["scada_ws"], bins=[0, 3, 6, 9, 12, 15, 100],
                                 labels=["0-3", "3-6", "6-9", "9-12", "12-15", "15+"])
    va_corr["abs_error"] = (va_corr["corrected_ws_v4"] - va_corr["scada_ws"]).abs()
    va_corr["signed_error"] = va_corr["corrected_ws_v4"] - va_corr["scada_ws"]

    print(f"===== {g} (과적합억제+구간가중치) =====")
    print(f"  TRAIN 15+구간 signed_error: {tr_signed_error:.3f} (n={len(tr_15plus)})")
    summary = va_corr.groupby("ws_bin", observed=True).agg(
        n=("scada_ws", "size"),
        scada_mean=("scada_ws", "mean"),
        corrected_mean=("corrected_ws_v4", "mean"),
        signed_error_mean=("signed_error", "mean"),
        abs_error_mean=("abs_error", "mean"),
    )
    print(summary.round(3))
    print()


# ============================================================
# corrected_ws_v4를 train/test 전체에 반영
# ============================================================
for g in TARGET_COLS:
    model = wind_correction_models_v4[g]
    train_datasets[g]["corrected_ws_v4"] = model.predict(train_datasets[g][RAW_FEATS])
    test_datasets[g]["corrected_ws_v4"] = model.predict(test_datasets[g][RAW_FEATS])


# ============================================================
# corrected_ws_v4 포함해서 최종 발전량 모델 재평가 (top2 둘 다)
# ============================================================
FEATS_WITH_V4 = RAW_FEATS + ["corrected_ws_v4"]

v4_results = []

for g in TARGET_COLS:
    df = train_datasets[g]
    tr = df[(df["forecast_kst_dtm"] >= TRAIN_PERIOD_START[g]) & (df["forecast_kst_dtm"] < CORR_VAL_START)]
    va = df[df["forecast_kst_dtm"] >= CORR_VAL_START].copy()
    capacity = CAPACITY_KWH[g]
    y_va = va["target"].to_numpy(dtype=float)

    for name in TOP2_MODELS[g]:
        model = make_model(name)
        model.fit(tr[FEATS_WITH_V4], tr["target"])
        pred = np.clip(model.predict(va[FEATS_WITH_V4]), 0, capacity)
        score, nmae, ficr = group_score(y_va, pred, capacity)
        print(f"[{g}] {name:15s} (corrected_ws_v4) -> score={score:.4f} nmae={nmae:.4f} ficr={ficr:.4f}")
        v4_results.append({"group": g, "model": name, "score": score, "nmae": nmae, "ficr": ficr})
    print()

v4_results_df = pd.DataFrame(v4_results)
print("=" * 70)
print("corrected_ws_v4(과적합억제+구간가중치) vs baseline 비교")
print("=" * 70)
compare_v4 = v4_results_df.merge(
    baseline_summary_df[["group", "model", "score"]].rename(columns={"score": "baseline_score"}),
    on=["group", "model"]
)
compare_v4["score_diff"] = compare_v4["score"] - compare_v4["baseline_score"]
print(compare_v4[["group", "model", "baseline_score", "score", "score_diff"]].to_string(index=False))

===== kpx_group_1 (과적합억제+구간가중치) =====
  TRAIN 15+구간 signed_error: -1.562 (n=127)
           n  scada_mean  corrected_mean  signed_error_mean  abs_error_mean
ws_bin                                                                     
0-3     1022       2.301           3.553              1.253           1.272
3-6     3178       4.465           4.687              0.222           0.921
6-9     2353       7.412           7.870              0.459           1.359
9-12    1508      10.257          10.238             -0.020           0.967
12-15    575      13.238          12.107             -1.131           1.581
15+      141      16.156          14.136             -2.020           2.328

===== kpx_group_2 (과적합억제+구간가중치) =====
  TRAIN 15+구간 signed_error: -1.896 (n=639)
           n  scada_mean  corrected_mean  signed_error_mean  abs_error_mean
ws_bin                                                                     
0-3     1005       2.317           3.588              1.271           1.277
3

In [37]:
# ============================================================
# 최종 제출 파일 생성
# 옵션A: 그룹별 확정 학습기간 ~ 2024 전체 포함
# 옵션B: 2024년만
# 보정풍속(corrected_ws_v4)까지 포함한 FEATS_WITH_V4 사용
# ============================================================

sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv", encoding="utf-8-sig")
sample_submission["forecast_kst_dtm"] = pd.to_datetime(sample_submission["forecast_kst_dtm"])

def make_submission(train_period_filter_func, filename):
    predictions = {}
    for g in TARGET_COLS:
        df = train_datasets[g]
        tr_full = train_period_filter_func(df, g)
        capacity = CAPACITY_KWH[g]

        # 그룹별 top2 앙상블(평균)로 최종 제출 - 개별 모델보다 안정적
        preds_list = []
        for name in TOP2_MODELS[g]:
            model = make_model(name)
            model.fit(tr_full[FEATS_WITH_V4], tr_full["target"])
            pred = np.clip(model.predict(test_datasets[g][FEATS_WITH_V4]), 0, capacity)
            preds_list.append(pred)

        final_pred = np.mean(preds_list, axis=0)
        final_pred = np.clip(final_pred, 0, capacity)
        predictions[g] = pd.Series(final_pred, index=test_datasets[g]["forecast_kst_dtm"].values)
        print(f"[{g}] n_train={len(tr_full)} 학습 완료 (top2 앙상블: {TOP2_MODELS[g]})")

    submission = sample_submission[["forecast_id", "forecast_kst_dtm"]].copy()
    for g in TARGET_COLS:
        submission[g] = submission["forecast_kst_dtm"].map(predictions[g])
    submission["forecast_kst_dtm"] = submission["forecast_kst_dtm"].dt.strftime("%Y-%m-%d %H:%M:%S")

    output_path = DATA_DIR.parent / filename
    submission.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"Saved: {output_path} | shape={submission.shape} | NaN={submission[TARGET_COLS].isna().sum().sum()}")
    print()
    return submission


# 옵션A: 그룹별 확정 시작점 ~ 2024 전체
def filter_A(df, g):
    return df[df["forecast_kst_dtm"] >= TRAIN_PERIOD_START[g]]

# 옵션B: 2024년만
def filter_B(df, g):
    return df[df["forecast_kst_dtm"] >= "2024-01-01"]

print("=" * 60)
print("옵션 A: 그룹별 확정 시작점 ~ 2024 전체 학습")
print("=" * 60)
submission_A = make_submission(filter_A, "submission_A_full_period.csv")

print("=" * 60)
print("옵션 B: 2024년만 학습")
print("=" * 60)
submission_B = make_submission(filter_B, "submission_B_2024only.csv")

옵션 A: 그룹별 확정 시작점 ~ 2024 전체 학습
[kpx_group_1] n_train=17536 학습 완료 (top2 앙상블: ['LightGBM', 'HistGB'])
[kpx_group_2] n_train=26201 학습 완료 (top2 앙상블: ['HistGB', 'LightGBM'])
[kpx_group_3] n_train=17538 학습 완료 (top2 앙상블: ['MLP', 'RandomForest'])
Saved: submission_A_full_period.csv | shape=(8760, 5) | NaN=0

옵션 B: 2024년만 학습
[kpx_group_1] n_train=8779 학습 완료 (top2 앙상블: ['LightGBM', 'HistGB'])
[kpx_group_2] n_train=8779 학습 완료 (top2 앙상블: ['HistGB', 'LightGBM'])
[kpx_group_3] n_train=8779 학습 완료 (top2 앙상블: ['MLP', 'RandomForest'])
Saved: submission_B_2024only.csv | shape=(8760, 5) | NaN=0



In [38]:
# ============================================================
# 옵션 C: 그룹1도 2022년부터 전부 포함 (그룹2·3은 옵션A와 동일)
# 옵션A(그룹1=2023부터)와 비교하기 위한 실험
# ============================================================

sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv", encoding="utf-8-sig")
sample_submission["forecast_kst_dtm"] = pd.to_datetime(sample_submission["forecast_kst_dtm"])

# 그룹1만 2022년부터, 나머지는 기존 TRAIN_PERIOD_START(그룹2=2022, 그룹3=2023) 그대로 유지
TRAIN_PERIOD_START_C = {
    "kpx_group_1": "2022-01-01",  # 원래 2023-01-01 이었던 것을 2022로 변경
    "kpx_group_2": TRAIN_PERIOD_START["kpx_group_2"],  # 2022-01-01, 동일
    "kpx_group_3": TRAIN_PERIOD_START["kpx_group_3"],  # 2023-01-01, 동일 (원래부터 이것뿐)
}

def filter_C(df, g):
    return df[df["forecast_kst_dtm"] >= TRAIN_PERIOD_START_C[g]]

predictions_C = {}
for g in TARGET_COLS:
    df = train_datasets[g]
    tr_full = filter_C(df, g)
    capacity = CAPACITY_KWH[g]

    preds_list = []
    for name in TOP2_MODELS[g]:
        model = make_model(name)
        model.fit(tr_full[FEATS_WITH_V4], tr_full["target"])
        pred = np.clip(model.predict(test_datasets[g][FEATS_WITH_V4]), 0, capacity)
        preds_list.append(pred)

    final_pred = np.mean(preds_list, axis=0)
    final_pred = np.clip(final_pred, 0, capacity)
    predictions_C[g] = pd.Series(final_pred, index=test_datasets[g]["forecast_kst_dtm"].values)
    print(f"[{g}] train_start={TRAIN_PERIOD_START_C[g]} n_train={len(tr_full)} 학습 완료 "
          f"(top2 앙상블: {TOP2_MODELS[g]})")

submission_C = sample_submission[["forecast_id", "forecast_kst_dtm"]].copy()
for g in TARGET_COLS:
    submission_C[g] = submission_C["forecast_kst_dtm"].map(predictions_C[g])
submission_C["forecast_kst_dtm"] = submission_C["forecast_kst_dtm"].dt.strftime("%Y-%m-%d %H:%M:%S")

output_path_C = DATA_DIR.parent / "submission_C_group1_full.csv"
submission_C.to_csv(output_path_C, index=False, encoding="utf-8-sig")
print(f"\nSaved: {output_path_C} | shape={submission_C.shape} | NaN={submission_C[TARGET_COLS].isna().sum().sum()}")

[kpx_group_1] train_start=2022-01-01 n_train=26200 학습 완료 (top2 앙상블: ['LightGBM', 'HistGB'])
[kpx_group_2] train_start=2022-01-01 n_train=26201 학습 완료 (top2 앙상블: ['HistGB', 'LightGBM'])
[kpx_group_3] train_start=2023-01-01 n_train=17538 학습 완료 (top2 앙상블: ['MLP', 'RandomForest'])

Saved: submission_C_group1_full.csv | shape=(8760, 5) | NaN=0
